<a href="https://colab.research.google.com/github/Kkurzo/-labs/blob/main/%22%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%9C%D0%B5%D0%BB%D1%8C%D0%BD%D0%B8%D1%87%D1%83%D0%BA_%D0%94%D0%BC%D0%B8%D1%82%D1%80%D0%BE_%D0%A4%D0%86%D0%A2_4_10_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Мельничук Дмитро
**Група:** ФІТ 4-10
**Дата:** 25.11.

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [13]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [18]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise
from sklearn.model_selection import train_test_split
from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [19]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [20]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print('Кількість унікальних користувачів:',
      ratings_df['user_id'].nunique())

print('Кількість унікальних фільмів:',
      ratings_df['item_id'].nunique())

print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', ratings_df['rating'].mean())

print('\nПропущені значення:')
print(ratings_df.isnull().sum())
display(ratings_df['rating'].value_counts().sort_index())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.52986

Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


,count
rating,
1.0,6110
2.0,11370
3.0,27145
4.0,34174
5.0,21201


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [21]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print('Навчальна вибірка:', len(train_df))
print('Тестова вибірка:', len(test_df))

print('Частка навчальної вибірки:',
      round(len(train_df) / len(ratings_df) * 100, 2), '%')

print('Частка тестової вибірки:',
      round(len(test_df) / len(ratings_df) * 100, 2), '%')

Навчальна вибірка: 80000
Тестова вибірка: 20000
Частка навчальної вибірки: 80.0 %
Частка тестової вибірки: 20.0 %


In [22]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(
        index=False,
        name=None
    )
)

print('Розмір навчальних даних:', len(train_data.raw_ratings))
print('Розмір тестових даних:', len(testset))

Розмір навчальних даних: 80000
Розмір тестових даних: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [25]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

print(param_grid_svd)
print(param_grid_knn)

{'n_factors': [50, 100], 'lr_all': [0.005, 0.01], 'reg_all': [0.02, 0.05]}
{'k': [20, 40], 'min_k': [1, 5], 'sim_options': {'name': ['cosine', 'pearson'], 'user_based': [False, True]}}


**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [27]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

In [ ]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

In [29]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:

print('\nМодель SVD:')
print('Найкращий RMSE:',
      grid_search_svd.best_score['rmse'])
print('Найкращий MAE:',
      grid_search_svd.best_score['mae'])
print('Найкращі параметри:',
      grid_search_svd.best_params['rmse'])

print('\nМодель KNNBasic:')
print('Найкращий RMSE:',
      grid_search_knn.best_score['rmse'])
print('Найкращий MAE:',
      grid_search_knn.best_score['mae'])
print('Найкращі параметри:',
      grid_search_knn.best_params['rmse'])

cv_results = pd.DataFrame([
    {
        'Model': 'SVD',
        'CV RMSE': grid_search_svd.best_score['rmse'],
        'CV MAE': grid_search_svd.best_score['mae'],
        'Best Parameters': str(
            grid_search_svd.best_params['rmse']
        )
    },
    {
        'Model': 'KNNBasic',
        'CV RMSE': grid_search_knn.best_score['rmse'],
        'CV MAE': grid_search_knn.best_score['mae'],
        'Best Parameters': str(
            grid_search_knn.best_params['rmse']
        )
    }
])

display(cv_results)


Модель SVD:
Найкращий RMSE: 0.9434401047887211
Найкращий MAE: 0.7437088409853451
Найкращі параметри: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

Модель KNNBasic:
Найкращий RMSE: 1.0264898935630498
Найкращий MAE: 0.8143933508016161
Найкращі параметри: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


,Model,CV RMSE,CV MAE,Best Parameters
0,SVD,0.94344,0.743709,"{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0..."
1,KNNBasic,1.02649,0.814393,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ..."


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [30]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(
    **best_svd_params,
    random_state=42
)

best_knn = KNNBasic(
    **best_knn_params,
    verbose=False
)

best_svd.fit(trainset)
best_knn.fit(trainset)

print('\nПараметри SVD:')
print(best_svd_params)

print('\nПараметри KNNBasic:')
print(best_knn_params)


Параметри SVD:
{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

Параметри KNNBasic:
{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [31]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print('\nSVD:')
print('RMSE:', round(svd_rmse, 4))
print('MAE:', round(svd_mae, 4))

print('\nKNNBasic:')
print('RMSE:', round(knn_rmse, 4))
print('MAE:', round(knn_mae, 4))


SVD:
RMSE: 0.9192
MAE: 0.7234

KNNBasic:
RMSE: 1.012
MAE: 0.7984


In [33]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

results = results.sort_values(by='RMSE')

print('Порівняння моделей на тестовій вибірці:')
display(results)

print('\nНайкраща модель за тестовим RMSE:',
      results.iloc[0]['Model'])

print('\nПорівняння CV та тестових результатів:')

comparison = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'CV RMSE': [
        grid_search_svd.best_score['rmse'],
        grid_search_knn.best_score['rmse']
    ],
    'Test RMSE': [svd_rmse, knn_rmse],
    'CV MAE': [
        grid_search_svd.best_score['mae'],
        grid_search_knn.best_score['mae']
    ],
    'Test MAE': [svd_mae, knn_mae]
})

display(comparison)

Порівняння моделей на тестовій вибірці:


,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421



Найкраща модель за тестовим RMSE: SVD

Порівняння CV та тестових результатів:


,Model,CV RMSE,Test RMSE,CV MAE,Test MAE
0,SVD,0.94344,0.919175,0.743709,0.723355
1,KNNBasic,1.02649,1.011953,0.814393,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [34]:
# Початковий ID можна змінити

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
user_id = '196'
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    recommendation_model = best_svd
    recommendation_name = 'SVD'
else:
    recommendation_model = best_knn
    recommendation_name = 'KNNBasic'

print('Обрана модель:', recommendation_name)
try:
    inner_uid = trainset.to_inner_uid(user_id)
    rated_items = set(
        trainset.to_raw_iid(inner_iid)
        for inner_iid, rating in trainset.ur[inner_uid]
    )
    all_items = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in trainset.all_items()
    ]
    unrated_items = [
        item_id for item_id in all_items
        if item_id not in rated_items
    ]
    recommendations = []

    for item_id in unrated_items:
        predicted_rating = recommendation_model.predict(
            uid=user_id,
            iid=item_id
        ).est

        recommendations.append({
            'item_id': item_id,
            'predicted_rating': predicted_rating
        })
    recommendations_df = pd.DataFrame(recommendations)

    recommendations_df = recommendations_df.sort_values(
        by='predicted_rating',
        ascending=False
    ).head(10).reset_index(drop=True)

    recommendations_df.index += 1

    print('\nТоп-10 рекомендацій для користувача', user_id)
    display(recommendations_df)

except ValueError:
    print('Користувача', user_id,
          'немає у навчальній вибірці.')

Обрана модель: SVD

Топ-10 рекомендацій для користувача 196


,item_id,predicted_rating
1,320,4.490358
2,64,4.470680
3,480,4.413710
4,313,4.401920
5,513,4.399624
6,483,4.398092
7,318,4.392102
8,656,4.385379
9,22,4.377388
10,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

У ході лабораторної роботи було досліджено рекомендаційні системи на основі алгоритмів SVD та KNNBasic. Для експерименту використано набір даних MovieLens 100K. Дані було розділено на навчальну та тестову вибірки у співвідношенні 80/20. За допомогою GridSearchCV підібрано оптимальні гіперпараметри обох моделей. Якість моделей оцінено за метриками RMSE та MAE.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**